# 12. Negocjacje agentów AI: uczenie preferencji drugiej strony

Agent A nie zna funkcji użyteczności B. Obserwuje wcześniejsze decyzje „akceptuję/odrzucam”, uczy prosty model logistyczny, a następnie wyszukuje oferty korzystne dla siebie i prawdopodobnie akceptowalne dla B.

**Założenie organizacyjne:** notebook działa lokalnie i nie wymaga internetu.

In [1]:
import itertools
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Trzy kwestie, każda ma poziom 0,1,2.
offers=np.array(list(itertools.product(range(3), repeat=3)),dtype=float)

# Jawna użyteczność A i ukryta użyteczność B.
wA=np.array([3.0, 1.0, 2.0])
wB=np.array([-1.0, 3.0, 2.5])

def utility(x,w):
    return float(x@w)

# B akceptuje, jeśli jego użyteczność przekracza próg, z niewielkim szumem decyzyjnym.
rng=np.random.default_rng(4)
train_idx=rng.integers(0,len(offers),size=220)
X=offers[train_idx]
latent=X@wB - 5.0
prob=1/(1+np.exp(-latent))
y=(rng.random(len(X))<prob).astype(int)

model=LogisticRegression().fit(X,y)
print('training accuracy=',round(accuracy_score(y,model.predict(X)),3))
print('learned coefficients=',model.coef_.round(2))

training accuracy= 0.868
learned coefficients= [[-0.9   2.63  2.24]]


In [2]:
p_accept=model.predict_proba(offers)[:,1]
uA=np.array([utility(x,wA) for x in offers])
uB=np.array([utility(x,wB) for x in offers])

# Agent A maksymalizuje własną użyteczność przy wymaganym P(akceptacji)>=0.65
mask=p_accept>=0.65
idx=np.argmax(np.where(mask,uA,-np.inf))
chosen=offers[idx]

print('Wybrana oferta:',chosen.astype(int))
print('uA=',uA[idx], 'p_accept=',round(p_accept[idx],3), 'ukryte uB=',uB[idx])

Wybrana oferta: [2 2 2]
uA= 12.0 p_accept= 0.961 ukryte uB= 9.0


In [3]:
df=pd.DataFrame({
    'issue1':offers[:,0].astype(int),
    'issue2':offers[:,1].astype(int),
    'issue3':offers[:,2].astype(int),
    'uA':uA,
    'true_uB':uB,
    'p_accept':p_accept
}).sort_values(['p_accept','uA'],ascending=False)
display(df.head(10))

,issue1,issue2,issue3,uA,true_uB,p_accept
8,0,2,2,6.0,11.0,0.993219
17,1,2,2,9.0,10.0,0.983544
26,2,2,2,12.0,9.0,0.960612
7,0,2,1,4.0,8.5,0.939882
5,0,1,2,5.0,8.0,0.913819
16,1,2,1,7.0,7.5,0.864492
14,1,1,2,8.0,7.0,0.812274
25,2,2,1,10.0,6.5,0.722478
23,2,1,2,11.0,6.0,0.638424
6,0,2,0,2.0,6.0,0.625300


## Zadanie
- Zmień próg minimalnego `p_accept` z `0.65` na `0.85`. Jaką cenę płaci A za większą pewność porozumienia?
- Dodaj koszt czasu negocjacji.
- Zmień prawdziwe preferencje B w połowie procesu i sprawdź, jak model staje się nieaktualny.

**Dyskusja:** w realnym systemie trzeba rozdzielić trzy warstwy: model preferencji drugiej strony, funkcję użyteczności własnego agenta oraz jawny protokół negocjacji. LLM może generować język oferty, ale nie powinien ukrywać tych reguł w nieaudytowalnym promptcie.